# Tokenize On-Policy DPO Dataset
Reads preference pairs from `datasets_dpo/on_policy/`, tokenizes chosen and rejected
sequences, and writes tokenized JSONL with filtering for short, balanced pairs.

Run `generate_dpo_pairs` first. Run `dpo_on_policy` after.

In [ ]:
import os
import json
from google.colab import drive
from tokenizers import Tokenizer

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
TOKENIZER_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "tokenizer.json")

# ==================== CONFIGURATION ====================
DATASET_NAME = "on_policy"
BLOCK_SIZE = 768
MIN_RESPONSE_TOKENS = 20

# Filtering (keeps pairs clean and balanced for small models)
MAX_RESPONSE_TOKENS = 250    # drop if either response exceeds this
MAX_TOTAL_TOKENS = 500       # drop if prefix + response exceeds this
MAX_LENGTH_RATIO = 2.0       # drop if chosen/rejected length ratio exceeds this

DPO_DIR = os.path.join(SPARKYLLM, "datasets_dpo", DATASET_NAME)
RAW_FILE = os.path.join(DPO_DIR, f"{DATASET_NAME}_raw.jsonl")
OUT_FILE = os.path.join(DPO_DIR, f"{DATASET_NAME}_tokenized.jsonl")
META_FILE = os.path.join(DPO_DIR, "tokenize_meta.json")

assert os.path.exists(RAW_FILE), f"Raw file not found: {RAW_FILE}\nRun generate_dpo_pairs first."

# Load tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
eot_id = tokenizer.token_to_id("<|endoftext|>")
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer: {vocab_size} tokens | EOT: {eot_id}")
print(f"Raw file: {RAW_FILE}")

In [ ]:
# ==================== FORMAT ADAPTER ====================

def on_policy_adapter(obj):
    """On-policy pairs: prompt is already in Alpaca format."""
    # prompt already includes 'Instruction: ...\nResponse: '
    return obj["prompt"], obj["chosen"], obj["rejected"]

adapter = on_policy_adapter
print(f"Using adapter: {DATASET_NAME}")

In [ ]:
# ==================== TOKENIZE ====================

written = 0
skipped_short = 0
skipped_response_too_long = 0
skipped_total_too_long = 0
skipped_length_ratio = 0
total_chosen_tokens = 0
total_rejected_tokens = 0
total_prefix_len = 0

with open(OUT_FILE, "w", encoding="utf-8") as f_out:
    with open(RAW_FILE, "r", encoding="utf-8") as f_in:
        for line_num, line in enumerate(f_in):
            obj = json.loads(line)
            prefix_text, chosen_text, rejected_text = adapter(obj)

            prefix_ids = tokenizer.encode(prefix_text).ids
            chosen_resp_ids = tokenizer.encode(chosen_text).ids
            rejected_resp_ids = tokenizer.encode(rejected_text).ids

            # Filter: minimum response length
            if len(chosen_resp_ids) < MIN_RESPONSE_TOKENS or len(rejected_resp_ids) < MIN_RESPONSE_TOKENS:
                skipped_short += 1
                continue

            # Filter: response too long
            if len(chosen_resp_ids) > MAX_RESPONSE_TOKENS or len(rejected_resp_ids) > MAX_RESPONSE_TOKENS:
                skipped_response_too_long += 1
                continue

            # Full sequences: prefix + response + EOT
            chosen_ids = prefix_ids + chosen_resp_ids + [eot_id]
            rejected_ids = prefix_ids + rejected_resp_ids + [eot_id]

            # Filter: total too long
            if len(chosen_ids) > MAX_TOTAL_TOKENS or len(rejected_ids) > MAX_TOTAL_TOKENS:
                skipped_total_too_long += 1
                continue

            # Filter: length ratio (prevents length exploitation)
            longer = max(len(chosen_resp_ids), len(rejected_resp_ids))
            shorter = max(len(chosen_resp_ids), len(rejected_resp_ids), 1)
            shorter = min(len(chosen_resp_ids), len(rejected_resp_ids))
            shorter = max(shorter, 1)  # avoid division by zero
            if longer / shorter > MAX_LENGTH_RATIO:
                skipped_length_ratio += 1
                continue

            out_obj = {
                "chosen_ids": chosen_ids,
                "rejected_ids": rejected_ids,
                "prefix_len": len(prefix_ids),
            }
            f_out.write(json.dumps(out_obj) + "\n")

            written += 1
            total_chosen_tokens += len(chosen_ids)
            total_rejected_tokens += len(rejected_ids)
            total_prefix_len += len(prefix_ids)

            if (line_num + 1) % 5000 == 0:
                print(f"  {line_num + 1:,} lines processed...")

avg_prefix = total_prefix_len / written if written else 0
avg_chosen = total_chosen_tokens / written if written else 0
avg_rejected = total_rejected_tokens / written if written else 0
total_skipped = skipped_short + skipped_response_too_long + skipped_total_too_long + skipped_length_ratio

print(f"\nDone!")
print(f"  {written:,} pairs written")
print(f"  {total_skipped:,} skipped:")
print(f"    too short:        {skipped_short:,}")
print(f"    response too long: {skipped_response_too_long:,}")
print(f"    total too long:   {skipped_total_too_long:,}")
print(f"    length ratio:     {skipped_length_ratio:,}")
print(f"  Avg prefix: {avg_prefix:.1f} | Avg chosen: {avg_chosen:.1f} | Avg rejected: {avg_rejected:.1f} tokens")
print(f"  Saved to: {OUT_FILE}")

# Save metadata
meta = {
    "dataset": DATASET_NAME,
    "num_pairs": written,
    "skipped_short": skipped_short,
    "skipped_response_too_long": skipped_response_too_long,
    "skipped_total_too_long": skipped_total_too_long,
    "skipped_length_ratio": skipped_length_ratio,
    "total_chosen_tokens": total_chosen_tokens,
    "total_rejected_tokens": total_rejected_tokens,
    "avg_prefix_len": round(avg_prefix, 1),
    "avg_chosen_len": round(avg_chosen, 1),
    "avg_rejected_len": round(avg_rejected, 1),
    "block_size": BLOCK_SIZE,
    "eot_id": eot_id,
    "vocab_size": vocab_size,
    "tokenizer_path": TOKENIZER_PATH,
    "filters": {
        "max_response_tokens": MAX_RESPONSE_TOKENS,
        "max_total_tokens": MAX_TOTAL_TOKENS,
        "max_length_ratio": MAX_LENGTH_RATIO,
    },
}
with open(META_FILE, "w") as f:
    json.dump(meta, f, indent=2)
print(f"  Meta: {META_FILE}")
print(f"\nNext step: run dpo_on_policy.ipynb")